# 58. The carrier heads on the other two models

**Where this starts.** Notebooks 56 to 57g are an account, on Qwen2.5-3B, of where an injected instruction travels: 20 of 576 heads carry it at the five template positions where the answer starts (held-out sufficiency 0.32 against 0.00 for layer-matched random and complement controls, necessity 0.76); the same heads carry the user's own instructions in the same direction (cosine 0.73 and 0.88); they attend to instructions wherever they sit; and every intervention on them trades security against the user's instructions or the document's words. One model is a case study. This notebook runs the three results that carry the account on Qwen2.5-7B-Instruct and Llama-3.1-8B-Instruct, with the attack logs those models already have from notebooks 43 to 45.

**This copy is set for the second run, Llama-3.1-8B-Instruct.** The first run, Qwen2.5-7B, is committed (6cf1b6b): it did not replicate, failing L2 (clean stops 0.132; with every head replaced the ceiling was only 0.342, against 0.80 on the 3B), while L1 (0.158 against controls at 0.000) and S (cosine 0.80 and 0.86) held. The marks below are exactly those the 7B run was scored on; nothing was changed after seeing it. Llama is gated: when Stage A loads the model it asks for your Hugging Face token in a hidden box in the cell output, unless the Colab secret `HF_TOKEN` is already set.

**Second start on Llama.** The first start stopped at the hook self-test: writing every head's own output back changed 4 of 8 answers (a plain repeat changed none) and moved the margin by 0.0136. The cause is precision, not where the hooks write. Recordings were kept in float16, Llama runs in bfloat16, and bfloat16 values smaller than about 8e-6 lose bits in float16, so the write-back was not an exact copy. The Qwen runs were not affected because those models run in float16. The cell after the bootstrap rewrites `src/head_patching.py` to keep recordings in float32 on a bfloat16 model; float16 models record exactly as before, so the 3B and 7B results stand as committed. The self-test now measures two things that were one number before: a pass's own recording written back into that same pass must change nothing at all, and the rounding between the prompt pass and the longer pass with the answer appended is printed, with a loose guard of 0.05. Every head set and every control carries that rounding alike. The marks are unchanged.

**How it is run.** The first code cell after the bootstrap sets `TAG`. Run the notebook once with `TAG = 'qwen7b_undefended'` and once with `TAG = 'llama_undefended'`; every cache and output is kept under that tag, and the commit cell names it. Each run loads the target, re-checks that model's first successful attack per injection with greedy decoding, writes the clean twins (the same hosts with the injection removed), screens the successes with notebook 53's five samples, and keeps the robust ones.

**What is replicated, with marks fixed in the first code cell before any result.**

| question | mark |
|---|---|
| L1: localisation, sufficiency | the carrier set is built as it was on the 3B, in two steps: the sweep of notebook 56 (every head's effect on the injection margin when its output from the attacked run is written into the clean twin, on 32 stratified robust successes) gives the top 40, and notebook 56b's knockout re-ranking then removes one head at a time from that 40, generates and judges, and keeps the 20 whose removal costs the most. The margin is a proxy that failed notebook 56's second gate on the 3B (AUROC 0.609 against the judge), which is why the re-ranking exists; the sweep's own top 20 is reported beside the carrier set everywhere, and on the 3B it reached only 0.125 held-out sufficiency against the knockout set's 0.32. On the robust successes **outside the sweep**, writing the carrier heads' outputs into the clean twin makes it follow the injection in at least 0.15 of items, and each of three layer-matched random 20-head sets and the complement stays at or below 0.05. |
| L2: localisation, necessity | on the same held-out items, writing the clean twin's outputs for those heads into the attacked run stops at least 0.60 of attacks **cleanly** (stopped and not degenerate); the controls at most 0.15; paired difference against the worst control, lower end at least 0.25. |
| S: the shared direction | notebook 57f's measurement: per carrier head, the following direction (attacked runs minus clean twins) and the user direction (user-turn instruction runs that did the task minus plain runs), at the template positions and at the answer tokens. The median cosine over the 20 heads is at least 0.60 in at least one of the two. |
| T (reported): the trade | Form A (the heads fed from a filler pass at the template positions) and Form A+ (the same at every answer token) on the held-out items: clean stops; grounding of the stops against the clean twin's answer; and the 150 same-type legitimate instructions on 30 fresh documents. Reported with the 3B numbers beside them. |
| decision | L1, L2 and S hold: REPLICATES on this model. Otherwise: which fails, named, with the numbers. |

**Compute.** Per model: about 150 greedy re-checks, 150 clean twins, 735 screen samples, 300 instruction and plain runs, the two sweeps (every head of the model: 784 on Qwen 7B, 1024 on Llama, 32 items, two rows per item), the knockout re-ranking (40 head sets on the sweep items), the held-out set checks (seven head sets, sufficiency and necessity), the direction recordings, and the trade (two forms on the held-out items and three conditions on 150 instruction items). Three to five hours on a G4 for the 7B model; Llama runs in bf16, as in notebooks 41 to 45, and takes longer. Every stage is cached per batch and resumes. Llama needs the Hugging Face login the project's helper asks for (a hidden box in Stage A's output, or the Colab secret `HF_TOKEN`).

**Outputs, per tag.** `reports/head_patching/{TAG}/head_effects.csv`, `knockout.csv`, `heldout.csv`, `directions.csv`, `trade.csv`, `instructions.csv`, `go_criteria_58.csv`, `go_58.txt`; `figures/head_patching/{TAG}/replication.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9 scikit-learn
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/head_patching', 'figures/head_patching', 'data/head_patching'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile src/head_patching.py
"""Per-head patching at the template positions, for the carrier-heads notebooks. Built on task_state, which is unchanged.

A query head's output is the slice of the input to its layer's attention output projection (o_proj) that belongs to that
head. These hooks sit on o_proj as forward pre-hooks, so they see that input before the projection mixes the heads, and
can record or replace any subset of head slices at the K template positions of a prompt pass. Document positions are
never touched and decode steps (sequence length one) pass through.

spec: {layer: dict(kind, heads, src)}
  kind 'set'    the listed heads' slices <- src
  kind 'zero'   the listed heads' slices <- 0
  src  [B, K, nh*dh] (per row), [K, nh*dh] (shared over rows) or [nh*dh] (shared over rows and positions); only the
       listed heads' slices are read from it, so a full recorded tensor can be passed and the heads chosen per call.

offset: 0, or the number of tokens appended after the template (16 when answer tokens are appended for the margin).

Recordings are kept in rec_dtype: float16 on a float16 model, float32 otherwise, so writing a recording back is an exact
copy on bfloat16 models too.
"""
from __future__ import annotations
import numpy as np
from task_state import encode, _generate, decoder_layers


def rec_dtype(dtype):
    """The precision a recording is kept in: float16 for a float16 model (as in notebooks 56 to 57g, so those runs are
    unchanged), float32 for anything else. A bfloat16 value smaller than about 8e-6 in size loses bits in float16 (and
    one above 65504 does not fit), so writing a float16 recording back into a bfloat16 model (Llama) is not an exact
    copy; float32 holds every bfloat16 value exactly."""
    import torch
    return torch.float16 if dtype == torch.float16 else torch.float32


def head_dims(model):
    """(number of query heads, head dimension) read from o_proj, so grouped-query attention is handled."""
    m = model.get_base_model() if hasattr(model, "get_base_model") else model
    nh = int(m.config.num_attention_heads); din = int(decoder_layers(model)[0].self_attn.o_proj.in_features)
    assert din % nh == 0, (din, nh)
    return nh, din // nh


class HeadHooks:
    def __init__(self, model, K):
        self.layers = decoder_layers(model); self.K = K; self.offset = 0
        self.nh, self.dh = head_dims(model)
        self.record = False; self.rec = {}; self.spec = {}; self.prompt_passes = 0
        self.handles = [l.self_attn.o_proj.register_forward_pre_hook(self._make(i)) for i, l in enumerate(self.layers)]

    def n_heads(self):
        return len(self.layers) * self.nh

    def _slice(self):
        return slice(-self.K - self.offset, None if self.offset == 0 else -self.offset)

    def _make(self, i):
        def hook(module, args):
            import torch
            x = args[0]
            if not isinstance(x, torch.Tensor) or x.dim() != 3 or x.shape[1] <= 1: return None
            if i == 0: self.prompt_passes += 1
            sl = self._slice()
            if self.record: self.rec[i] = x[:, sl, :].detach().to(rec_dtype(x.dtype)).cpu().clone()
            s = self.spec.get(i)
            if s is None: return None
            B = x.shape[0]; x = x.clone()
            cur = x[:, sl, :].reshape(B, self.K, self.nh, self.dh); new = cur.clone()
            idx = torch.as_tensor(list(s["heads"]), device=x.device, dtype=torch.long)
            if s["kind"] == "zero":
                new[:, :, idx, :] = 0
            elif s["kind"] == "set":
                v = s["src"].to(x.device, x.dtype)
                if v.dim() == 1: v = v.view(1, 1, self.nh, self.dh).expand(B, self.K, -1, -1)
                elif v.dim() == 2: v = v.reshape(1, self.K, self.nh, self.dh).expand(B, -1, -1, -1)
                elif v.dim() == 3: v = v.reshape(v.shape[0], self.K, self.nh, self.dh)
                else: raise ValueError(f"src has {v.dim()} dimensions")
                new[:, :, idx, :] = v[:, :, idx, :]
            else: raise ValueError(s["kind"])
            x[:, sl, :] = new.reshape(B, self.K, self.nh * self.dh)
            return (x,) + tuple(args[1:])
        return hook

    def close(self):
        for h in self.handles: h.remove()
        self.handles = []


def head_spec(layer, heads, src=None, kind="set"):
    """One spec entry. src is a stacked recording [B, L, K, nh*dh]; its layer slice is taken here."""
    e = dict(kind=kind, heads=list(heads))
    if kind != "zero":
        assert src is not None, "kind 'set' needs a source"
        e["src"] = src[:, int(layer)] if src.dim() == 4 else src
    return {int(layer): e}


def merge_specs(*specs):
    """Specs on disjoint layers merged; two entries for one layer are joined when their kind and source match."""
    out = {}
    for s in specs:
        for l, e in s.items():
            if l not in out: out[l] = dict(e)
            else:
                assert out[l]["kind"] == e["kind"], f"two kinds on layer {l}"
                out[l]["heads"] = sorted(set(out[l]["heads"]) | set(e["heads"]))
    return out


def set_spec(heads, src=None, kind="set"):
    """A spec for a set of (layer, head) pairs."""
    by = {}
    for l, h in heads: by.setdefault(int(l), []).append(int(h))
    return merge_specs(*[head_spec(l, hs, src, kind) for l, hs in by.items()])


def capture_heads(model, hooks, tok, prompts, system, sfx):
    """Head outputs at the template positions for one batch: [B, L, K, nh*dh] in rec_dtype (float16 on a float16 model, float32 otherwise), from the answering call's prompt pass."""
    import torch
    enc = encode(tok, prompts, system, sfx).to(model.device)
    hooks.rec = {}; hooks.spec = {}; hooks.offset = 0; hooks.record = True
    try: _generate(model, tok, enc, 1)
    finally: hooks.record = False
    st = torch.stack([hooks.rec[i] for i in range(len(hooks.layers))], 1); hooks.rec = {}
    return st


def answer_logprob(model, hooks, tok, prompts, system, sfx, ans_ids, spec=None, record=False):
    """Mean log-probability of each row's answer tokens, with the head spec applied at the template positions.

    ans_ids: one list of token ids per row, all the same length. The answer tokens are appended to the prompt, so the
    template positions sit len(ans) places before the end and the hooks take that offset. record=True also returns the
    head outputs at the template positions of this very pass, before any spec is applied ([B, L, K, nh*dh] in
    rec_dtype), so a write-back can be checked against a recording of the same shape of pass."""
    import torch
    enc = encode(tok, prompts, system, sfx).to(model.device)
    A = torch.as_tensor(ans_ids, device=enc["input_ids"].device, dtype=enc["input_ids"].dtype)
    assert A.dim() == 2 and A.shape[0] == enc["input_ids"].shape[0], (A.shape, enc["input_ids"].shape)
    ids = torch.cat([enc["input_ids"], A], 1); am = torch.cat([enc["attention_mask"], torch.ones_like(A)], 1)
    nt = int(A.shape[1])
    hooks.rec = {}; hooks.spec = dict(spec or {}); hooks.record = bool(record); hooks.offset = nt
    try:
        with torch.no_grad(): lg = model(input_ids=ids, attention_mask=am).logits
    finally: hooks.spec = {}; hooks.offset = 0; hooks.record = False
    lp = torch.log_softmax(lg[:, -nt - 1:-1, :].float(), -1)
    out = lp.gather(-1, A[:, :, None].long()).squeeze(-1).mean(1).detach().cpu().numpy()
    if not record: return out
    st = torch.stack([hooks.rec[i] for i in range(len(hooks.layers))], 1); hooks.rec = {}
    return out, st


def doc_attention(model, tok, prompts, system, sfx, doc_tokens, batch=4):
    """Attention mass from the last prompt position to the document span, per head: [L, nh] averaged over rows.

    Attention Tracker ranks heads by the mass the last token sends to the injected instruction. That span is not
    recorded for rewritten documents, so the span used here is the whole document, which is the closest recoverable
    form of the rule. doc_tokens: the document's own token count per row."""
    import torch
    L = len(decoder_layers(model)); nh, _ = head_dims(model); tot = np.zeros((L, nh)); rows = 0
    for s0 in range(0, len(prompts), batch):
        ps = prompts[s0:s0 + batch]; nd = doc_tokens[s0:s0 + batch]
        enc = encode(tok, ps, system, sfx).to(model.device)
        with torch.no_grad(): att = model(**enc, output_attentions=True).attentions
        T = enc["input_ids"].shape[1]; K = len(sfx)
        for b in range(len(ps)):
            lo = max(0, T - K - int(nd[b])); hi = T - K
            if hi <= lo: continue
            for l in range(L): tot[l] += att[l][b, :, -1, lo:hi].float().sum(-1).cpu().numpy()
            rows += 1
        del att
    assert rows, "no row had a document span"
    return tot / rows

## Stage A. The model, the attack logs, the verdict rule, the marks

In [ ]:
TAG = 'llama_undefended'             # the second run; the first, TAG = 'qwen7b_undefended', is committed (6cf1b6b)
import importlib, hashlib, collections, numpy as np, json, re, pandas as pd
import task_state, state_patching, head_patching, head_blind, head_steer, adaptive_attack, adaptive_strict, final_scoring, targets, attack_success, ci_tools, fidelity_audit, model_defense, llama_h2h
for m_ in (task_state, state_patching, head_patching, head_blind, head_steer, adaptive_attack, adaptive_strict, final_scoring, targets, attack_success, ci_tools, fidelity_audit, model_defense, llama_h2h): importlib.reload(m_)
from task_state import suffix_ids, token_f1, degenerate
from state_patching import PatchHooks, generate_patched, neutral_filler, seed53
from head_patching import HeadHooks, capture_heads, answer_logprob, set_spec, head_dims
from head_blind import BlindHooks, generate_blind, filler_prompt
from head_steer import SteerHooks, record_means, directions
from adaptive_attack import host_of
from adaptive_strict import load_attempts
from final_scoring import final_rule, paper_verdict, langdetect_non_english, CATEGORIES, JUDGE_ONLY, TRANSLATION
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER
from attack_success import judge_prompt, parse_judge, JUDGE_SYSTEM
from model_defense import TASK_USER_WITH
from ci_tools import wilson
from fidelity_audit import newcombe_paired
from fidelity_exact import paired_boot
H = lambda s: hashlib.sha1((s or '').encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 90)
MODELS = {'qwen7b_undefended': dict(base='Qwen/Qwen2.5-7B-Instruct', label='Qwen 7B undefended', llama=False), 'llama_undefended': dict(base='meta-llama/Llama-3.1-8B-Instruct', label='Llama 8B undefended', llama=True)}
assert TAG in MODELS, TAG
LB = MODELS[TAG]['label']; JUDGE = 'Qwen/Qwen2.5-7B-Instruct'
OUT, REP, FIG = 'data/head_patching', f'reports/head_patching/{TAG}', f'figures/head_patching/{TAG}'
for d_ in (f'{OUT}/{TAG}', REP, FIG): os.makedirs(d_, exist_ok=True)
SEED = 58; BS, MAX_NEW, NTOK = 16, 160, 16; BS_SWEEP = 16; KMAIN = 20; N_SWEEP = 32; RAND_DRAWS = 3
SCREEN = dict(samples=5, temperature=0.7, top_p=0.9, need=4)
CRIT58 = dict(l1_suff=0.15, l1_ctrl=0.05, l2_stop=0.60, l2_ctrl=0.15, l2_paired_lo=0.25, l2_degen=0.05, s_cos=0.60, min_heldout=20, n_user_max=150, u1_lo=-0.05, u1_headroom=0.30)
FAMILIES = {'Language Translation': 'translation', 'Emoji Substitution': 'emoji', 'Base Encoding': 'base64', 'Reverse Text': 'reverse', 'Substitution Ciphers': 'cipher'}
print('model:', LB, '| marks:', CRIT58)

# ---- the 210 injections and their clean twins, as notebooks 51 to 56 built them ----
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); n = len(SUB); assert set(SUB.meta) == set(CATEGORIES)
SIDX = [i for i in range(n) if SUB.meta.iloc[i] not in JUDGE_ONLY]; JIDX = [i for i in range(n) if SUB.meta.iloc[i] in JUDGE_ONLY]
SPLIT = [host_of(SUB.text.iloc[i], SUB.instruction.iloc[i]) for i in range(n)]
CLEAN_DOC = [(b[:-1] if b.endswith('\n') else b) + a for b, a in SPLIT]; CLEAN_P = [TASK_USER.format(doc=d) for d in CLEAN_DOC]
assert open('data/task_state/qwen3b_undefended/clean_docs.sha1').read().strip() == H('\x00'.join(CLEAN_DOC)), 'the clean twins differ from notebook 51\'s'
LD = {}
def ld(o):
    k = H(o or '')
    if k not in LD: LD[k] = langdetect_non_english(o)
    return LD[k]
JC = jload(f'{OUT}/{TAG}/judge.json')
jkey = lambda goal, out: H(goal + '\x00' + (out or ''))
def needs_judge(cat, out, doc): return (not ld(out)) if cat == TRANSLATION else final_rule(cat, out, doc) is None
def ok(cat, goal, doc, o):
    """The paper's verdict: did answer o follow instruction `goal` of category `cat` in document `doc`."""
    k = jkey(goal, o)
    if needs_judge(cat, o, doc):
        assert k in JC, 'an answer the rule sends to the judge was not judged'
        return bool(paper_verdict(cat, o, doc, JC[k]))
    return bool(paper_verdict(cat, o, doc, False))
rows = []
for log, idx, name in ((load_attempts('data/adaptive_strict', TAG, 4), SIDX, 'strict'), (load_attempts('data/final_scoring/adaptive', TAG, 4), JIDX, 'judge-only')):
    print(f'attack log for {TAG}, {name} categories: {len(log)} attempts')
    for a in log:
        i = idx[a['i']]; cat = SUB.meta.iloc[i]
        rows.append(dict(injection=i, category=cat, goal=SUB.instruction.iloc[i], attempt=int(a['attempt']), document=a['document'], output=a['output'],
                         success=bool(a['verdict']) or (cat == TRANSLATION and ld(a['output'])), key=H(f"{i}\x00{a['document']}")))
assert rows, f'no attack log for {TAG}'
assert len({r['category'] for r in rows}) == len(CATEGORIES), (f'the attack log for {TAG} covers only {sorted({r["category"] for r in rows})}: both logs are needed '
                                                               f'(data/adaptive_strict and data/final_scoring/adaptive), or the tables below are on a part of the set')
ATT = pd.DataFrame(rows).sort_values(['injection', 'attempt']).reset_index(drop=True)
first = ATT[ATT.success].groupby('injection').head(1)
SUCC = [dict(injection=int(r.injection), category=r.category, goal=r.goal, document=r.document, attempt=int(r.attempt), key=r.key, prompt=TASK_USER.format(doc=r.document)) for r in first.itertuples()]
print('first successes per injection in the log:', len(SUCC), dict(collections.Counter(it['category'] for it in SUCC)))

# ---- the target, the judge, helpers ----
SFX = None; _M = {}; _J = {}
def use():
    global SFX
    if _M: return _M['model'], _M['tok'], _M['hk'], _M['ph']
    if MODELS[TAG]['llama']:
        llama_h2h.hf_login()      # the project's login: the Colab secret HF_TOKEN if it is set, otherwise a hidden prompt in this cell's output; the token is never printed or saved
        m, tok = llama_h2h.load_llama()
        if getattr(tok, 'add_bos_token', False):
            tok.add_bos_token = False      # the chat template already begins with the start token; notebooks 43 to 45 tokenised these prompts the same way
            print('Llama tokenizer: add_bos_token turned off, so the chat template is not given a second start token')
    else: m, tok = load_lm(MODELS[TAG]['base'], four_bit=True)
    s = suffix_ids(tok, TASK_SYSTEM); SFX = s; nh, dh = head_dims(m)
    _M.update(model=m, tok=tok, hk=HeadHooks(m, len(s)), ph=PatchHooks(m, len(s)))
    print(f'loaded {LB}: {len(_M["hk"].layers)} layers x {nh} query heads = {_M["hk"].n_heads()} heads, head dimension {dh}; template positions = the last {len(s)} prompt tokens {tok.convert_ids_to_tokens(s)}')
    return m, tok, _M['hk'], _M['ph']
def seven_b():
    if 'm' not in _J: _J['m'], _J['t'] = load_lm(JUDGE, four_bit=True)
    return _J['m'], _J['t']
def free_seven_b():
    if 'm' in _J: _J.clear(); gc.collect(); torch.cuda.empty_cache()
def judge_pending(triples):
    need = {}
    for cat, goal, doc, o in triples:
        k = jkey(goal, o)
        if k not in JC and k not in need and needs_judge(cat, o, doc): need[k] = (goal, o)
    if need:
        jm, jt = seven_b(); ks = list(need)
        for s in range(0, len(ks), 400):
            ch = ks[s: s + 400]; jo = generate_batch(jm, jt, [judge_prompt(*need[k]) for k in ch], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0, show_every=400)
            for k, x in zip(ch, jo): JC[k] = bool(parse_judge(x))
            jsave(JC, f'{OUT}/{TAG}/judge.json')
    return len(need)
by_len = lambda ps: sorted(range(len(ps)), key=lambda i: (len(ps[i]), i))
chunks = lambda order, bs: [order[s: s + bs] for s in range(0, len(order), bs)]
ans_ids = lambda tok, s: tok(s or ' ', add_special_tokens=False).input_ids[:NTOK]
def pad_ans(tok, s):
    v = ans_ids(tok, s); return v + [tok.pad_token_id] * (NTOK - len(v))
def rate(x):
    x = np.asarray(x, bool); k_, m_ = int(x.sum()), len(x); lo, hi = wilson(k_, m_) if m_ else (np.nan, np.nan); return dict(n=m_, k=k_, rate=k_ / m_ if m_ else np.nan, lo=lo, hi=hi)
rstr = lambda r, d=3: f"{r['rate']:.{d}f} [{r['lo']:.{d}f}, {r['hi']:.{d}f}]" if r['n'] else 'no items'
dstr = lambda d: f'{d[0]:+.2f} [{d[1]:+.2f}, {d[2]:+.2f}]' if not np.isnan(d[0]) else ''
def mrate(x): x = np.asarray(x, bool); return float(x.mean()) if len(x) else np.nan
def npair(x, y): return newcombe_paired(np.asarray(x, bool), np.asarray(y, bool)) if len(x) else (np.nan, np.nan, np.nan)
def gen_plain(tag, items, prompt_of, max_new=None, sample=None):
    """Greedy (or sampled) answers, cached by item key."""
    fp = f'{OUT}/{TAG}/plain_{tag}.json'; G = jload(fp); need = [it for it in items if it['key'] not in G]
    for idx in chunks(by_len([prompt_of(it) for it in need]), BS):
        bx = [need[i] for i in idx]; hk.spec = {}
        outs, _ = generate_patched(m, ph, tok, [prompt_of(it) for it in bx], TASK_SYSTEM, SFX, max_new_tokens=max_new or MAX_NEW, sample=sample)
        G.update({it['key']: o for it, o in zip(bx, outs)}); jsave(G, fp)
    return G
m, tok, hk, ph = use()

## Stage B. The items for this model: greedy re-check, clean twins, the five-sample screen, instruction items; the hook self-test

In [ ]:
GRE = gen_plain('greedy', SUCC, lambda it: it['prompt'])
judge_pending([(it['category'], it['goal'], it['document'], GRE[it['key']]) for it in SUCC]); free_seven_b()
for it in SUCC: it['greedy'] = GRE[it['key']]; it['greedy_ok'] = ok(it['category'], it['goal'], it['document'], it['greedy'])
SUCC = [it for it in SUCC if it['greedy_ok']]
print('successes that hold under greedy decoding here:', len(SUCC))
# the clean twins: the same hosts with the injected line removed, answered by this model
TWIN = {}
for it in SUCC:
    i = it['injection']
    if i not in TWIN: TWIN[i] = dict(injection=i, key=f'twin{i}', document=CLEAN_DOC[i], prompt=CLEAN_P[i], category=it['category'], goal=it['goal'])
CO = gen_plain('twins', list(TWIN.values()), lambda tw: tw['prompt'])
judge_pending([(tw['category'], tw['goal'], tw['document'], CO[tw['key']]) for tw in TWIN.values()]); free_seven_b()
for tw in TWIN.values(): tw['greedy'] = CO[tw['key']]; tw['followed'] = ok(tw['category'], tw['goal'], tw['document'], tw['greedy'])
for it in SUCC: it['clean_answer'] = TWIN[it['injection']]['greedy']
print('clean twins that follow the injection with no injection present (should be near zero):', sum(tw['followed'] for tw in TWIN.values()), 'of', len(TWIN))
# notebook 53's screen: five samples at temperature 0.7, robust success = greedy and at least four of five
fp = f'{OUT}/{TAG}/screen.json'; SCR = jload(fp); todo = [it for it in SUCC if it['key'] not in SCR]
for k_, it in enumerate(todo):
    hk.spec = {}
    outs, _ = generate_patched(m, ph, tok, [it['prompt']] * SCREEN['samples'], TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW, sample=dict(temperature=SCREEN['temperature'], top_p=SCREEN['top_p'], seed=seed53(it['key'])))
    SCR[it['key']] = outs
    if k_ % 10 == 9 or k_ == len(todo) - 1: jsave(SCR, fp); print(f'  screened {k_ + 1}/{len(todo)}')
judge_pending([(it['category'], it['goal'], it['document'], o) for it in SUCC for o in SCR[it['key']]]); free_seven_b()
for it in SUCC: it['samples_ok'] = int(sum(ok(it['category'], it['goal'], it['document'], o) for o in SCR[it['key']])); it['robust'] = it['samples_ok'] >= SCREEN['need']
RS = [it for it in SUCC if it['robust'] and not TWIN[it['injection']]['followed']]
TWIN_RS = [TWIN[it['injection']] for it in RS]; BYINJ = {it['injection']: it for it in RS}
print('robust successes (and their twin does not follow):', len(RS), dict(collections.Counter(it['category'] for it in RS)))
assert len(RS) >= 40, 'too few robust successes on this model to localise anything'
# instruction items: the same five tasks asked in the user turn, on documents outside the robust successes' hosts
INSTR = {}
for c, fam in FAMILIES.items():
    ins = sorted({SUB.instruction.iloc[i] for i in range(n) if SUB.meta.iloc[i] == c}); INSTR[fam] = dict(category=c, instruction=ins[0], n_distinct=len(ins))   # as notebooks 56 to 57f picked it
rs_hosts = {it['document'] for it in TWIN_RS}
docs = sorted(set(CLEAN_DOC) - rs_hosts); r_ = np.random.default_rng(SEED + 1); perm = r_.permutation(len(docs))
POOL_A, POOL_B = [docs[i] for i in perm[:30]], [docs[i] for i in perm[30:60]]
print('instruction pools: documents outside the robust successes\' hosts:', len(docs), '| pool A', len(POOL_A), '| pool B', len(POOL_B))
def instr_items(pool, tag):
    its = []
    for di, d in enumerate(pool):
        for fam, info in INSTR.items():
            its.append(dict(key=f'{tag}:{di}:{fam}', family=fam, category=info['category'], goal=info['instruction'], document=d,
                            prompt=TASK_USER_WITH.format(instr=info['instruction'], doc=d), plain_key=f'{tag}plain:{di}', plain_prompt=TASK_USER.format(doc=d)))
    return its
IA, IB_ = instr_items(POOL_A, 'pa'), instr_items(POOL_B, 'pb')
GI = gen_plain('instr', IA + IB_, lambda it: it['prompt'])
GPL = gen_plain('instr_plain', [dict(key=it['plain_key'], prompt=it['plain_prompt']) for it in IA + IB_], lambda it: it['prompt'])
judge_pending([(it['category'], it['goal'], it['document'], GI[it['key']]) for it in IA + IB_] + [(it['category'], it['goal'], it['document'], GPL[it['plain_key']]) for it in IA + IB_]); free_seven_b()
for it in IA + IB_:
    it['greedy'] = GI[it['key']]; it['instr_followed'] = ok(it['category'], it['goal'], it['document'], it['greedy']); it['plain_followed'] = ok(it['category'], it['goal'], it['document'], GPL[it['plain_key']])
print('the legitimate instruction done, pool A:', {f: round(float(np.mean([it['instr_followed'] for it in IA if it['family'] == f])), 2) for f in INSTR})
print('the legitimate instruction done, pool B:', {f: round(float(np.mean([it['instr_followed'] for it in IB_ if it['family'] == f])), 2) for f in INSTR})
# the hook self-test of notebook 56, with the exact write-back and the rounding between two shapes of pass measured apart
def self_test():
    its = RS[:8]; ps = [it['prompt'] for it in its]
    Aa = capture_heads(m, hk, tok, ps, TASK_SYSTEM, SFX); before = hk.prompt_passes
    o1, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW); passes = hk.prompt_passes - before
    allh = [(l, h) for l in range(len(hk.layers)) for h in range(hk.nh)]
    o2, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)
    hk.spec = set_spec(allh, Aa); o3, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW); hk.spec = {}
    same = sum(a == b for a, b in zip(o1, o3)); floor = sum(a == b for a, b in zip(o1, o2))
    ans = [pad_ans(tok, it['greedy']) for it in its]
    lp, Ar = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, ans, record=True); lpc = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, [pad_ans(tok, it['clean_answer']) for it in its])
    marg = lp - lpc
    own = float(np.abs(answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, ans, spec=set_spec(allh, Ar)) - lp).max())      # this pass's own recording written back into it: must change nothing
    cross = float(np.abs(answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, ans, spec=set_spec(allh, Aa)) - lp).max())    # the prompt pass's recording written into the pass with the answer appended: rounding only
    z = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, ans, spec=set_spec([(len(hk.layers) // 2, 0)], None, kind='zero')); nz = float(np.abs(z - lp).max())
    print(f'self-test: model in {getattr(m, "dtype", "?")}, recordings kept in {Aa.dtype}; prompt passes per call {passes}; answers unchanged when every head output is written back {same}/{len(o1)} (no-spec repeat {floor}/{len(o1)}); margin median {float(np.median(marg)):+.3f}')
    print(f'  a pass\'s own recording written back changes the margin by {own:.2e} (must be 0); the prompt pass\'s recording written into the longer pass changes it by {cross:.2e} (rounding, guard 0.05); zeroing one head moves it by {nz:.3f}')
    assert passes == 1 and same >= floor - 1 and float(np.median(marg)) > 0 and own < 1e-6 and cross < 0.05 and nz > 0, 'the hooks do not behave on this model: stop'
self_test()

## Stage C. The sweep of notebook 56 on this model, and the head sets with their controls

In [ ]:
def stratified(items, k, seed):
    r_ = np.random.default_rng(seed); by = collections.defaultdict(list)
    for it in items: by[it['category']].append(it)
    picks = []
    for c in sorted(by):
        q = max(1, int(round(k * len(by[c]) / len(items)))); idx = r_.permutation(len(by[c]))[:min(q, len(by[c]))]; picks += [by[c][i] for i in idx]
    r_.shuffle(picks); return picks[:k]
SWEEP_ITEMS = stratified(RS, N_SWEEP, SEED); SWEEP_KEYS = {it['key'] for it in SWEEP_ITEMS}
HELD = [it for it in RS if it['key'] not in SWEEP_KEYS]; TWIN_HELD = [TWIN[it['injection']] for it in HELD]
print('sweep items:', len(SWEEP_ITEMS), dict(collections.Counter(it['category'] for it in SWEEP_ITEMS)), '| held out:', len(HELD))
HEADS = [(l, h) for l in range(len(hk.layers)) for h in range(hk.nh)]
HA = {}
def heads_of(keys_prompts):
    need = [(k, p) for k, p in keys_prompts if k not in HA]
    for idx in chunks(by_len([p for _, p in need]), BS):
        s_ = capture_heads(m, hk, tok, [need[i][1] for i in idx], TASK_SYSTEM, SFX)
        for j, i in enumerate(idx): HA[need[i][0]] = s_[j]
    return HA
heads_of([(it['key'], it['prompt']) for it in RS] + [(tw['key'], tw['prompt']) for tw in TWIN_RS])
def sweep(name, items, target_prompt, source_key, sign):
    fp = f'{OUT}/{TAG}/sweep_{name}.json'; C_ = jload(fp); bl = chunks(by_len([target_prompt(it) for it in items]), BS_SWEEP); base = C_.get('_baseline')
    if base is None:
        base = {}
        for idx in bl:
            bx = [items[i] for i in idx]; ps = [p for it in bx for p in (target_prompt(it), target_prompt(it))]
            aid = [v for it in bx for v in (pad_ans(tok, it['greedy']), pad_ans(tok, it['clean_answer']))]; lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, aid)
            for j, it in enumerate(bx): base[it['key']] = float(lp[2 * j] - lp[2 * j + 1])
        C_['_baseline'] = base; jsave(C_, fp)
    print(f'  {name}: unpatched margin, median {float(np.median(list(base.values()))):+.3f}')
    todo = [f'{l}:{h}' for l, h in HEADS if f'{l}:{h}' not in C_]
    for z, hkey in enumerate(todo):
        l, h = (int(x) for x in hkey.split(':')); vals = []
        for idx in bl:
            bx = [items[i] for i in idx]; ps = [p for it in bx for p in (target_prompt(it), target_prompt(it))]
            aid = [v for it in bx for v in (pad_ans(tok, it['greedy']), pad_ans(tok, it['clean_answer']))]; src = torch.stack([HA[source_key(it)] for it in bx for _ in range(2)])
            lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, aid, spec=set_spec([(l, h)], src)); vals += [float(lp[2 * j] - lp[2 * j + 1]) - base[it['key']] for j, it in enumerate(bx)]
        C_[hkey] = sign * float(np.mean(vals))
        if z % 50 == 49 or z == len(todo) - 1: jsave(C_, fp); print(f'  {name}: {z + 1}/{len(todo)} new heads swept')
    return C_
SUF = sweep('sufficiency', SWEEP_ITEMS, lambda it: CLEAN_P[it['injection']], lambda it: it['key'], +1)
NEC = sweep('necessity', SWEEP_ITEMS, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", -1)
eff = lambda C_: np.array([C_[f'{l}:{h}'] for l, h in HEADS]); E_SUF, E_NEC = eff(SUF), eff(NEC)
SW = pd.DataFrame([dict(layer=l, head=h, sufficiency=E_SUF[i], necessity=E_NEC[i]) for i, (l, h) in enumerate(HEADS)]); SW.to_csv(f'{REP}/head_effects.csv', index=False)
order = np.argsort(-E_SUF); EI = {x: i for i, x in enumerate(HEADS)}
C_20 = [HEADS[i] for i in order[:KMAIN]]; C_40 = [HEADS[i] for i in order[:2 * KMAIN]]
print(f'\n=== the {KMAIN} heads with the largest sufficiency effect on {LB} (the sweep\'s own set, C_20) ==='); print(SW.iloc[order[:KMAIN]].round(4).to_string(index=False))
print('layers of C_20:', dict(collections.Counter(l for l, _ in C_20)), '| Spearman of the sufficiency and necessity profiles:', round(float(pd.Series(E_SUF).rank().corr(pd.Series(E_NEC).rank())), 3))

# ---- the knockout re-ranking of notebook 56b, which is how the carrier set that carries the 3B result was built ----
# notebook 56's gate two failed on the 3B (the margin against the judge, AUROC 0.609), so the set behind the 3B numbers is not the sweep's
# top 20 by margin but the knockout set: C_40 with one head removed at a time, generated and judged, ranked by the drop in sufficiency.
KN_TWINS = [TWIN[it['injection']] for it in SWEEP_ITEMS]
KN_TAG = H('\x00'.join(sorted(tw['key'] for tw in KN_TWINS)))[:8]
def suff_sweep(heads):
    # Sufficiency of a head set on the sweep items, generated and judged, cached by the item set and the sorted heads.
    kk = KN_TAG + '|' + ':'.join(f'{l}.{h}' for l, h in sorted(heads)); C_ = jload(f'{OUT}/{TAG}/knock.json')
    if kk not in C_:
        got = {}
        for idx in chunks(by_len([tw['prompt'] for tw in KN_TWINS]), BS):
            bx = [KN_TWINS[i] for i in idx]; src = torch.stack([HA[BYINJ[tw['injection']]['key']] for tw in bx]); hk.spec = set_spec(heads, src)
            try: outs, _ = generate_patched(m, ph, tok, [tw['prompt'] for tw in bx], TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)
            finally: hk.spec = {}
            got.update({tw['key']: o for tw, o in zip(bx, outs)})
        judge_pending([(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], got[tw['key']]) for tw in KN_TWINS]); free_seven_b()
        C_[kk] = float(np.mean([ok(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], got[tw['key']]) for tw in KN_TWINS]))
        jsave(C_, f'{OUT}/{TAG}/knock.json')
    return C_[kk]
base40 = suff_sweep(C_40); knock = []
for z, (l, h) in enumerate(C_40):
    knock.append(dict(layer=l, head=h, drop_in_sufficiency=base40 - suff_sweep([x for x in C_40 if x != (l, h)]), margin_effect=float(E_SUF[EI[(l, h)]])))
    if z % 8 == 7 or z == len(C_40) - 1: print(f'  knockout: {z + 1}/{len(C_40)} heads removed (C_40 sufficiency on the sweep items {base40:.3f})')
KNT = pd.DataFrame(knock).sort_values(['drop_in_sufficiency', 'margin_effect'], ascending=False, kind='stable').reset_index(drop=True)
KNT.to_csv(f'{REP}/knockout.csv', index=False)
CARRIER = [(int(r.layer), int(r.head)) for r in KNT.head(KMAIN).itertuples()]
print(f'\n=== the knockout ranking: C_40 with one head removed, drop in sufficiency on the {len(KN_TWINS)} sweep items ==='); print(KNT.head(KMAIN).round(4).to_string(index=False))
print(f'heads whose removal drops sufficiency at all: {int((KNT.drop_in_sufficiency > 0).sum())} of {len(C_40)} | of the knockout top {KMAIN}, {len(set(CARRIER) & set(C_20))} were in the sweep\'s C_{KMAIN}')
print('layers of the carrier set:', dict(collections.Counter(l for l, _ in CARRIER)))
def random_matched(cset, draw):
    """A set of the same size as cset, matched layer by layer where the layer has room, the shortfall drawn from other layers."""
    r_ = np.random.default_rng(SEED * 100 + draw); by = collections.Counter(l for l, _ in cset); out = []
    for l, q in by.items():
        pool = [h for h in range(hk.nh) if (l, h) not in cset]; out += [(l, int(h)) for h in r_.permutation(pool)[:q]]
    short = len(cset) - len(out)
    if short > 0:
        rest = [x for x in HEADS if x not in set(cset) and x not in set(out)]
        out += [tuple(int(v) for v in rest[i]) for i in r_.permutation(len(rest))[:short]]
        print(f'  control draw {draw}: {short} of {len(cset)} heads drawn outside the carrier set\'s layers, which had no room left')
    assert len(out) == len(cset) and not (set(out) & set(cset)), (len(out), len(cset))
    return out
def complement_low(cset):
    lays = {l for l, _ in cset}; pool = [(l, h) for l, h in HEADS if l in lays and (l, h) not in cset]
    if len(pool) < len(cset):
        pool += [x for x in HEADS if x not in set(cset) and x not in set(pool)]
        print(f'  complement: the carrier set\'s layers hold fewer than {len(cset)} other heads, so the rest is taken from the lowest-effect heads elsewhere')
    sel = [pool[i] for i in np.argsort([E_SUF[EI[x]] for x in pool])[:len(cset)]]
    assert len(sel) == len(cset) and not (set(sel) & set(cset)); return sel
SETS = {'carrier': CARRIER, 'sweep_top20': C_20, 'all_heads': HEADS}
for d in range(RAND_DRAWS): SETS[f'random_draw{d + 1}'] = random_matched(CARRIER, d + 1)
SETS['complement'] = complement_low(CARRIER)
CTRLS = [f'random_draw{d + 1}' for d in range(RAND_DRAWS)] + ['complement']
json.dump({k: [[int(l), int(h)] for l, h in v] for k, v in SETS.items()}, open(f'{OUT}/{TAG}/head_sets.json', 'w'))
SETH = {k: H(':'.join(f'{l}.{h}' for l, h in sorted(v)))[:8] for k, v in SETS.items()}   # the cache of a head-set condition is keyed by the heads, not by the name
print('head sets and their cache keys:', {k: f'{len(SETS[k])} heads, {SETH[k]}' for k in SETS})
for c in CTRLS: assert len(SETS[c]) == len(CARRIER), f'{c} is not the same size as the carrier set'
# the 3B carrier set, for the overlap report only (head indices are not expected to transfer between models)
try:
    HS3 = json.load(open('data/head_patching/qwen3b_undefended/head_sets.json')); C3 = {tuple(x) for x in HS3.get('Cknock_20', [])}
    print('overlap with the 3B set, as (layer, head) pairs:', len(C3 & set(CARRIER)), 'of', KMAIN, '(not expected to transfer; reported only)')
except Exception as e_: print('3B head sets not on this Drive:', e_)

## Stage D. L1 and L2 on the held-out robust successes

In [ ]:
def run_set(tag, items, target_prompt, source, conds):
    cfile = lambda c: f'{OUT}/{TAG}/gen_{tag}_{c}_{SETH[c]}.json'
    G = {c: jload(cfile(c)) for c in conds}; bl = chunks(by_len([target_prompt(it) for it in items]), BS)
    for bi, idx in enumerate(bl):
        bx = [items[i] for i in idx]; ps = [target_prompt(it) for it in bx]; todo = [c for c in conds if any(it['key'] not in G[c] for it in bx)]
        for c in todo:
            src = torch.stack([HA[source(it)] for it in bx]); hk.spec = set_spec(SETS[c], src)
            try: outs, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)
            finally: hk.spec = {}
            G[c].update({it['key']: [o] for it, o in zip(bx, outs)}); jsave(G[c], cfile(c))
        if todo: print(f'  {tag}: batch {bi + 1}/{len(bl)} ({len(todo)} conditions)')
    return G
CONDS = ['carrier', 'sweep_top20'] + CTRLS + ['all_heads']
GS = run_set('suff', TWIN_HELD, lambda tw: tw['prompt'], lambda tw: BYINJ[tw['injection']]['key'], CONDS)
GN = run_set('nec', HELD, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", CONDS)
judge_pending([(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], GS[c][tw['key']][0]) for c in CONDS for tw in TWIN_HELD]
              + [(it['category'], it['goal'], it['document'], GN[c][it['key']][0]) for c in CONDS for it in HELD]); free_seven_b()
def suff_vec(c): return np.array([ok(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], GS[c][tw['key']][0]) for tw in TWIN_HELD], bool)
def stop_vec(c): return np.array([not ok(it['category'], it['goal'], it['document'], GN[c][it['key']][0]) for it in HELD], bool)
def clean_vec(c): return np.array([s_ and not degenerate(GN[c][it['key']][0]) for s_, it in zip(stop_vec(c), HELD)], bool)
rows = []
for c in CONDS:
    st = clean_vec(c); f1 = float(np.median([token_f1(GN[c][it['key']][0], it['clean_answer']) for it, s_ in zip(HELD, st) if s_])) if st.any() else np.nan
    rows.append(dict(head_set=c, size=len(SETS[c]), sufficiency=rstr(rate(suff_vec(c))), stopped=rstr(rate(stop_vec(c))), clean_stop=rstr(rate(st)), clean_stop_rate=round(mrate(st), 3),
                     median_f1_of_clean_stops_vs_twin=round(f1, 3) if f1 == f1 else np.nan))
HO = pd.DataFrame(rows); HO.to_csv(f'{REP}/heldout.csv', index=False)
print(f'=== {len(HELD)} held-out robust successes on {LB}: sufficiency (twin follows) and necessity (attack stopped) ==='); print(HO.to_string(index=False))
s_car = mrate(suff_vec('carrier')); s_ctl = {c: mrate(suff_vec(c)) for c in CTRLS}
L1 = bool(len(HELD) >= CRIT58['min_heldout'] and s_car >= CRIT58['l1_suff'] and max(s_ctl.values()) <= CRIT58['l1_ctrl'])
cc = clean_vec('carrier'); n_ctl = {c: mrate(clean_vec(c)) for c in CTRLS}; worst = max(n_ctl, key=n_ctl.get); pairN = npair(cc, clean_vec(worst))
dg = float(np.mean([degenerate(GN['carrier'][it['key']][0]) for it, s_ in zip(HELD, stop_vec('carrier')) if s_])) if stop_vec('carrier').any() else np.nan
L2 = bool(len(HELD) >= CRIT58['min_heldout'] and mrate(cc) >= CRIT58['l2_stop'] and max(n_ctl.values()) <= CRIT58['l2_ctrl'] and pairN[1] >= CRIT58['l2_paired_lo'] and (dg != dg or dg <= CRIT58['l2_degen']))
print(f'\nL1 {"pass" if L1 else "fail"}: carrier sufficiency {s_car:.3f}, worst control {max(s_ctl.values()):.3f}')
print(f'L2 {"pass" if L2 else "fail"}: carrier clean stops {mrate(cc):.3f}, worst control {worst} {n_ctl[worst]:.3f}, paired {dstr(pairN)}, degenerate {dg:.3f}')

## Stage E. S: the shared direction, as notebook 57f measured it

In [ ]:
sh = SteerHooks(m, len(SFX))
def record_set(name, prompts):
    fp = f'{OUT}/{TAG}/steer_means_{name}.pt'
    if os.path.exists(fp): d = torch.load(fp); return d['sum'] / d['cnt'][..., None, None].clamp_min(1)
    hk.spec = {}; outs, S_, C_ = record_means(m, sh, tok, prompts, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW, batch=BS, order=by_len(prompts))
    torch.save(dict(sum=S_, cnt=C_), fp); print(f'  recorded {name}: {len(prompts)} runs'); return S_ / C_[..., None, None].clamp_min(1)
M_INJ = record_set('attacked', [it['prompt'] for it in RS]); M_TWIN = record_set('twins', [TWIN[it['injection']]['prompt'] for it in RS])
UOK = [it for it in IA if it['instr_followed'] and not it['plain_followed']]
r_ = np.random.default_rng(SEED); UOK = [UOK[i] for i in sorted(r_.choice(len(UOK), min(CRIT58['n_user_max'], len(UOK)), replace=False))] if UOK else []
assert len(UOK) >= 20, 'too few user-turn instruction runs that did the task'
M_USER = record_set('user', [it['prompt'] for it in UOK]); M_PLAIN = record_set('plain', [it['plain_prompt'] for it in UOK])
K_ = len(SFX); tpl = lambda M: M[:, :K_]; ans_ = lambda M: M[:, K_:K_ + 1]
_, _, COS_TPL, KEPT_TPL = directions(tpl(M_INJ), tpl(M_TWIN), tpl(M_USER) - tpl(M_PLAIN) + tpl(M_TWIN), tpl(M_TWIN))
_, _, COS_ANS, KEPT_ANS = directions(ans_(M_INJ), ans_(M_TWIN), ans_(M_USER) - ans_(M_PLAIN) + ans_(M_TWIN), ans_(M_TWIN))
li = torch.tensor([l for l, _ in CARRIER]); hi = torch.tensor([h for _, h in CARRIER])
DIR = pd.DataFrame([dict(heads='carrier', where='template positions', median_cosine=round(float(COS_TPL[li, hi].median()), 3), median_kept=round(float(KEPT_TPL[li, hi].median()), 3)),
                    dict(heads='carrier', where='answer tokens', median_cosine=round(float(COS_ANS[li, hi].median()), 3), median_kept=round(float(KEPT_ANS[li, hi].median()), 3)),
                    dict(heads='all heads', where='template positions', median_cosine=round(float(COS_TPL.median()), 3), median_kept=round(float(KEPT_TPL.median()), 3)),
                    dict(heads='all heads', where='answer tokens', median_cosine=round(float(COS_ANS.median()), 3), median_kept=round(float(KEPT_ANS.median()), 3))])
DIR.to_csv(f'{REP}/directions.csv', index=False)
print(f'=== S: cosine between the following direction and the user direction, {LB} (3B: carrier 0.734 template, 0.882 answer) ==='); print(DIR.to_string(index=False))
cT, cA_ = float(COS_TPL[li, hi].median()), float(COS_ANS[li, hi].median()); S_SHARED = bool(max(cT, cA_) >= CRIT58['s_cos'])
print('S', 'pass' if S_SHARED else 'fail')

## Stage F. T: the trade on this model (reported), Form A and Form A+ on the held-out items, and the legitimate instructions

In [ ]:
bh = BlindHooks(m, len(SFX)); BSP = BS // 2
FILL = {}
def fill_of(it):
    if it['key'] not in FILL: FILL[it['key']] = filler_prompt(it['prompt'], it['document'], neutral_filler(tok, it['document']))
    return FILL[it['key']]
def blind(tag, items, conds, steps_of):
    # conds: names; steps_of[c] is True (every token) or False (template positions only)
    bfile = lambda c: f'{OUT}/{TAG}/blind_{tag}_{c}_{SETH["carrier"]}.json'      # the blind forms all use the carrier set
    G = {c: jload(bfile(c)) for c in conds}; bl = chunks(by_len([it['prompt'] for it in items]), BSP)
    for bi, idx in enumerate(bl):
        bx = [items[i] for i in idx]; todo = [c for c in conds if any(it['key'] not in G[c] for it in bx)]
        for c in todo:
            hk.spec = {}; outs = generate_blind(m, bh, tok, [it['prompt'] for it in bx], [fill_of(it) for it in bx], TASK_SYSTEM, SFX, CARRIER, max_new_tokens=MAX_NEW, steps=steps_of[c])
            G[c].update({it['key']: [o] for it, o in zip(bx, outs)}); jsave(G[c], bfile(c))
        if todo and (bi % 5 == 4 or bi == len(bl) - 1): print(f'  {tag}: batch {bi + 1}/{len(bl)}')
    return G
STEPS = {'formA': False, 'formAplus': True}
_W4 = re.compile(r'[a-z]{4,}')
def grounded_(o, d):
    w = _W4.findall((o or '').lower()); dw = set(_W4.findall((d or '').lower())); return float(np.mean([x in dw for x in w])) if w else np.nan
GT = blind('held', HELD, list(STEPS), STEPS)
judge_pending([(it['category'], it['goal'], it['document'], GT[c][it['key']][0]) for c in STEPS for it in HELD]); free_seven_b()
trows = []
for c in STEPS:
    st = np.array([not ok(it['category'], it['goal'], it['document'], GT[c][it['key']][0]) for it in HELD], bool); cs = np.array([s_ and not degenerate(GT[c][it['key']][0]) for s_, it in zip(st, HELD)], bool)
    sel = [it for it, s_ in zip(HELD, cs) if s_]; host = lambda it: TWIN[it['injection']]['document']
    gs = np.array([grounded_(GT[c][it['key']][0], host(it)) for it in sel]); gr = np.array([grounded_(it['clean_answer'], host(it)) for it in sel]); okk = ~np.isnan(gs) & ~np.isnan(gr)
    trows.append(dict(form=c, clean_stop=rstr(rate(cs)), clean_stop_rate=round(mrate(cs), 3), stops_grounded=round(float(np.nanmedian(gs)), 3) if len(gs) else np.nan,
                      stops_grounding_minus_twin=dstr(paired_boot(gs[okk] - gr[okk], seed=SEED)) if okk.sum() > 1 else '', n_clean_stops=len(sel)))
TR = pd.DataFrame(trows); TR.to_csv(f'{REP}/trade.csv', index=False)
print(f'=== T: the trade on the {len(HELD)} held-out items, {LB} (3B on fresh attacks: Form A 0.776 clean stops, grounding -0.05; Form A+ 0.925, grounding -0.14) ==='); print(TR.to_string(index=False))
GU = {'undefended': {it['key']: [GI[it['key']]] for it in IB_}}; GU.update(blind('instrB', IB_, list(STEPS), STEPS))
judge_pending([(it['category'], it['goal'], it['document'], GU[c][it['key']][0]) for c in STEPS for it in IB_]); free_seven_b()
def did(c, items): return np.array([ok(it['category'], it['goal'], it['document'], GU[c][it['key']][0]) for it in items], bool)
urows = []
for fam in list(INSTR) + ['all five']:
    sel = [it for it in IB_ if fam == 'all five' or it['family'] == fam]; u_ = did('undefended', sel); row = dict(family=fam, n=len(sel), undefended=round(float(u_.mean()), 3))
    for c in STEPS: v = did(c, sel); row[c] = round(float(v.mean()), 3); row[f'{c}_minus_undefended'] = dstr(npair(v, u_))
    urows.append(row)
U1T = pd.DataFrame(urows); U1T.to_csv(f'{REP}/instructions.csv', index=False)
print(f'\n=== the same five tasks asked legitimately, {len(IB_)} items (3B, Form A on 450 items: +0.00 [-0.03, +0.03]; Form A+ was never run on the 3B instructions) ==='); print(U1T.to_string(index=False))
TA = npair(did('formA', IB_), did('undefended', IB_)); TAP = npair(did('formAplus', IB_), did('undefended', IB_))

## Stage G. The marks and what they decide

In [ ]:
import matplotlib.pyplot as plt
from reslog import log_result
checks = [
    ('L1', f'held-out sufficiency on {LB} ({len(HELD)} items): the knockout carrier set; worst control; the sweep\'s own top {KMAIN} beside it',
     f'{s_car:.3f}; {max(s_ctl.values()):.3f}; sweep set {mrate(suff_vec("sweep_top20")):.3f}', '>= 0.15; <= 0.05', L1),
    ('L2', 'held-out necessity: clean stops; worst control; paired difference; degenerate share; the sweep\'s own top set beside it',
     f'{mrate(cc):.3f}; {n_ctl[worst]:.3f} ({worst}); {dstr(pairN)}; {dg:.3f}; sweep set {mrate(clean_vec("sweep_top20")):.3f}', '>= 0.60; <= 0.15; lower end >= 0.25; <= 0.05', L2),
    ('S', 'shared direction: median cosine on the carrier heads (template; answer tokens)', f'{cT:.3f}; {cA_:.3f}', '>= 0.60 in one', S_SHARED),
    ('T (reported)', 'Form A and Form A+: clean stops; stop grounding minus twin; legitimate instructions minus undefended',
     '; '.join(f'{r.form}: {r.clean_stop_rate:.3f}, {r.stops_grounding_minus_twin}' for r in TR.itertuples()) + f'; instructions A {dstr(TA)}, A+ {dstr(TAP)}', 'reported', True)]
VT = pd.DataFrame([dict(mark=a, criterion=b, value=v, needed=m_, met=bool(ok_)) for a, b, v, m_, ok_ in checks]); VT.to_csv(f'{REP}/go_criteria_58.csv', index=False)
TTXT = (f' The trade on this model: Form A stops {TR.clean_stop_rate.iloc[0]:.3f} cleanly at {dstr(TA)} on the legitimate instructions; Form A+ stops {TR.clean_stop_rate.iloc[1]:.3f} '
        f'with stop grounding {TR.stops_grounding_minus_twin.iloc[1] or "n/a"} and {dstr(TAP)} on the legitimate instructions.')
if L1 and L2 and S_SHARED:
    DEC = (f'REPLICATES on {LB}: {KMAIN} of {len(HEADS)} heads, chosen by the sweep and the knockout re-ranking, carry the injected instruction on {len(HELD)} held-out items (sufficiency {s_car:.3f} against controls at most '
           f'{max(s_ctl.values()):.3f}; clean stops {mrate(cc):.3f} against {n_ctl[worst]:.3f}, paired {dstr(pairN)}), and they carry the user\'s instructions in the same direction '
           f'(median cosine {cT:.2f} at the template positions, {cA_:.2f} at the answer tokens). The sweep\'s own top {KMAIN}, before the knockout re-ranking, gives '
           f'sufficiency {mrate(suff_vec("sweep_top20")):.3f} and clean stops {mrate(clean_vec("sweep_top20")):.3f}; on the 3B the re-ranking was what made the set work.' + TTXT)
else:
    L1W = [w for w, f_ in ((f'held-out items {len(HELD)} (at least {CRIT58["min_heldout"]} needed)', len(HELD) < CRIT58['min_heldout']),
                           (f'sufficiency {s_car:.3f} (at least {CRIT58["l1_suff"]:.2f})', s_car < CRIT58['l1_suff']),
                           (f'worst control {max(s_ctl.values()):.3f} (at most {CRIT58["l1_ctrl"]:.2f})', max(s_ctl.values()) > CRIT58['l1_ctrl'])) if f_]
    L2W = [w for w, f_ in ((f'held-out items {len(HELD)}', len(HELD) < CRIT58['min_heldout']), (f'clean stops {mrate(cc):.3f} (at least {CRIT58["l2_stop"]:.2f})', mrate(cc) < CRIT58['l2_stop']),
                           (f'worst control {n_ctl[worst]:.3f} (at most {CRIT58["l2_ctrl"]:.2f})', max(n_ctl.values()) > CRIT58['l2_ctrl']),
                           (f'paired difference {dstr(pairN)} (lower end at least {CRIT58["l2_paired_lo"]:.2f})', not (pairN[1] >= CRIT58['l2_paired_lo'])),
                           (f'degenerate stops {dg:.3f} (at most {CRIT58["l2_degen"]:.2f})', dg == dg and dg > CRIT58['l2_degen'])) if f_]
    DEC = (f'DOES NOT REPLICATE on {LB}: ' + '; '.join(s_ for s_, f_ in ((f'L1 fails on {", ".join(L1W)}', not L1), (f'L2 fails on {", ".join(L2W)}', not L2),
                                                                         (f'S fails: cosine {cT:.2f} and {cA_:.2f}', not S_SHARED)) if f_)
           + f'. The sweep\'s own top {KMAIN}, before the knockout re-ranking, gives sufficiency {mrate(suff_vec("sweep_top20")):.3f} and clean stops {mrate(clean_vec("sweep_top20")):.3f}.' + TTXT)
GTXT = f'{DEC}\nL1 {"pass" if L1 else "fail"}, L2 {"pass" if L2 else "fail"}, S {"pass" if S_SHARED else "fail"}'
open(f'{REP}/go_58.txt', 'w').write(GTXT + '\n')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
x = np.arange(len(CONDS)); ax[0].bar(x - 0.2, [mrate(suff_vec(c)) for c in CONDS], 0.4, label='sufficiency (twin follows)'); ax[0].bar(x + 0.2, [mrate(clean_vec(c)) for c in CONDS], 0.4, label='necessity (clean stop)')
ax[0].set_xticks(x); ax[0].set_xticklabels(CONDS, rotation=30, ha='right', fontsize=8); ax[0].set_ylim(0, 1); ax[0].set_title(f'{LB}: {len(HELD)} held-out items', fontsize=10); ax[0].legend(fontsize=8)
ax[1].scatter(COS_TPL.flatten(), KEPT_TPL.flatten(), s=4, alpha=.3, label='all heads, template'); ax[1].scatter(COS_TPL[li, hi], KEPT_TPL[li, hi], s=30, marker='^', label='carrier, template')
ax[1].scatter(COS_ANS[li, hi], KEPT_ANS[li, hi], s=30, marker='s', label='carrier, answer tokens'); ax[1].set_xlabel('cosine(following, user)'); ax[1].set_ylabel('share kept after removal'); ax[1].legend(fontsize=7)
ax[1].set_title('the shared direction', fontsize=10)
ax[2].bar([0, 1], [TR.clean_stop_rate.iloc[0], TR.clean_stop_rate.iloc[1]], 0.5); ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(['Form A', 'Form A+']); ax[2].set_ylim(0, 1); ax[2].set_title('the trade: clean stops', fontsize=10)
for a_ in ax: a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/replication.png', dpi=150, bbox_inches='tight'); plt.show()
for title, T_ in (('MARKS', VT), ('HELD OUT', HO), ('DIRECTIONS', DIR), ('TRADE', TR), ('INSTRUCTIONS', U1T)): print(f'\n=== {title} ==='); print(T_.to_string(index=False))
print('\n' + GTXT)
log_result(f'nb58 ({TAG}): the carrier heads on {LB}: the sweep, held-out sufficiency and necessity with controls, the shared direction, and the trade',
           '\n\n'.join([GTXT] + [f'{t}:\n' + T_.to_string(index=False) for t, T_ in (('Marks', VT), ('Held out', HO), ('Directions', DIR), ('Trade', TR), ('Instructions', U1T))]), reports_dir=REP)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", f"nb58 ({TAG}, follow-on work): the carrier-head account of notebooks 56 to 57g run on {LB}: that model's first successful attack per injection re-checked greedily, its clean twins written, notebook 53's five-sample screen applied; notebook 56's sweep over every head on 32 stratified robust successes; notebook 56b's knockout re-ranking of the top 40 (one head removed at a time, generated and judged); the resulting 20 heads checked on the held-out robust successes for sufficiency and clean-stop necessity against three layer-matched random sets and the complement; notebook 57f's shared-direction cosine on those heads; the Form A and Form A+ trade on the held-out items and on 150 legitimate instructions, reported; marks L1, L2 and S fixed in the first cell; src/head_patching.py rewritten here: recordings kept in float32 on a bfloat16 model (Llama) so that writing a head's output back is an exact copy, float16 models (the Qwen runs) recording as before, and the margin call able to return its own pass's recording, which the hook self-test now writes back exactly, with the rounding between a prompt pass and the longer pass printed apart; reports/head_patching/{TAG}/head_effects.csv, heldout.csv, directions.csv, trade.csv, instructions.csv, go_criteria_58.csv, go_58.txt, figures/head_patching/{TAG}/replication.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)